# Seleksi Model dan Uji Function Calling

**Penelitian:** Evaluasi Agentic AI pada Tugas dan Bahasa Domain Lokal  
**Tahap:** Pemeriksaan awal kemampuan function calling sebelum evaluasi menggunakan MASSIVE-Agents.

Penelitian menggunakan tiga model berikut:

1. Qwen2.5-7B-Instruct
2. Gemma 2 9B Instruct
3. Sahabat-AI v1 9B Instruct

Pengujian dilakukan melalui Ollama secara lokal, dengan dua pendekatan:

- **Prompt JSON:** model diminta memilih fungsi dan mengisi argumen dalam format JSON.
- **Native tool calling:** fungsi dikirim melalui parameter `tools`, kemudian keluaran diperiksa pada `message.tool_calls`.

Pengujian awal menggunakan dua fungsi contoh dan perintah berbahasa Inggris serta Indonesia. Hasil tahap ini digunakan untuk memeriksa kompatibilitas model, **bukan untuk menghitung akurasi akhir MASSIVE-Agents**.


## 1. Konfigurasi

Ketiga model diuji dengan pengaturan yang sama. Model dijalankan bergantian agar penggunaan memori tidak terlalu besar. Waktu yang tercatat mencakup proses pemuatan model, sehingga belum digunakan sebagai perbandingan kecepatan inferensi.


In [2]:
import csv
import json
import platform
import time
import urllib.error
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Markdown, display

API_BASE = "http://127.0.0.1:11434"
MODELS = [
    "qwen2.5:7b-instruct",
    "gemma2:9b-instruct-q4_K_M",
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
]

OPTIONS = {"temperature": 0, "num_ctx": 2048, "num_predict": 128}
TIMEOUT_SECONDS = 600
RUN_JSON_MODE = True
RUN_NATIVE_MODE = True
# Uji native menggunakan dua perintah terkait lampu.

print("Python:", platform.python_version())
print("Model kandidat:", len(MODELS))
print("Konfigurasi:", OPTIONS)

Python: 3.13.12
Model kandidat: 3
Konfigurasi: {'temperature': 0, 'num_ctx': 2048, 'num_predict': 128}


## 2. Pemeriksaan Ollama

Memeriksa koneksi ke server lokal dan memastikan ketiga model tersedia sebelum pengujian dimulai.


In [3]:
def ollama_request(endpoint, payload=None, timeout=TIMEOUT_SECONDS):
    url = API_BASE + endpoint
    body = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=body,
        method="GET" if payload is None else "POST",
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {e.code} dari Ollama: {detail[:500]}") from e
    except urllib.error.URLError as e:
        raise RuntimeError(
            "Server Ollama tidak dapat dihubungi di 127.0.0.1:11434. "
            "Pastikan terminal `ollama serve` masih aktif."
        ) from e

version = ollama_request("/api/version")
tags = ollama_request("/api/tags")
installed = {x["name"].lower(): x for x in tags.get("models", [])}
print("Ollama version:", version.get("version", "tidak terbaca"))
print("\nModel tersedia:")
for name in MODELS:
    detail = installed.get(name.lower())
    print("  [OK]" if detail else "  [BELUM ADA]", name)

missing = [name for name in MODELS if name.lower() not in installed]
if missing:
    raise RuntimeError("Model belum terdaftar di Ollama: " + ", ".join(missing))

Ollama version: 0.40.13

Model tersedia:
  [OK] qwen2.5:7b-instruct
  [OK] gemma2:9b-instruct-q4_K_M
  [OK] Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m


## 3. Fungsi dan data uji

Pengujian awal menggunakan dua fungsi sederhana, yaitu mengendalikan lampu dan meminta informasi cuaca. Setiap fungsi diuji dengan perintah bahasa Inggris dan Indonesia yang sepadan.

Fungsi ini hanya digunakan untuk pengujian teknis. Evaluasi utama nantinya menggunakan definisi fungsi dari dataset MASSIVE-Agents.


In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "control_light",
            "description": "Turn a light in a room on or off / Nyalakan atau matikan lampu ruangan",
            "parameters": {
                "type": "object",
                "properties": {
                    "room": {"type": "string", "description": "Room name mentioned by the user"},
                    "action": {"type": "string", "enum": ["on", "off"]},
                },
                "required": ["room", "action"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a city / Dapatkan informasi cuaca kota",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City requested by the user"},
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    },
]

CASES = [
    {"case_id":"EN_LIGHT", "language":"en-US", "prompt":"Turn on the bedroom light.",
     "expected_name":"control_light", "expected_arguments":{"room":"bedroom", "action":"on"}},
    {"case_id":"ID_LIGHT", "language":"id-ID", "prompt":"Tolong nyalakan lampu kamar tidur.",
     "expected_name":"control_light", "expected_arguments":{"room":"kamar tidur", "action":"on"}},
    {"case_id":"EN_WEATHER", "language":"en-US", "prompt":"What's the weather in Jakarta?",
     "expected_name":"get_weather", "expected_arguments":{"city":"Jakarta"}},
    {"case_id":"ID_WEATHER", "language":"id-ID", "prompt":"Bagaimana cuaca di Jakarta?",
     "expected_name":"get_weather", "expected_arguments":{"city":"Jakarta"}},
]

print(f"Fungsi contoh: {len(TOOLS)}")
for t in CASES:
    print(f"{t['case_id']:12s} {t['prompt']}")

Fungsi contoh: 2
EN_LIGHT     Turn on the bedroom light.
ID_LIGHT     Tolong nyalakan lampu kamar tidur.
EN_WEATHER   What's the weather in Jakarta?
ID_WEATHER   Bagaimana cuaca di Jakarta?


## 4. Metode pengujian dan penilaian

Setiap prediksi diperiksa berdasarkan:

1. Format keluaran dapat dibaca.
2. Nama fungsi sesuai dengan perintah.
3. Nama argumen sesuai dengan fungsi.
4. Nilai argumen sesuai dengan jawaban yang diharapkan.

Pada **prompt JSON**, hasil dibaca dari `message.content`. Pada **native tool calling**, hasil harus tersedia di `message.tool_calls`. Kedua pendekatan dihitung secara terpisah.

Pemeriksaan nilai argumen menggunakan kecocokan teks setelah penyeragaman huruf besar-kecil. Perbedaan sinonim atau terjemahan tetap memerlukan pemeriksaan lebih lanjut.


In [5]:
def build_payload(model, case, mode):
    base = {
        "model": model,
        "messages": [],
        "stream": False,
        "options": OPTIONS,
        "keep_alive": 0,
    }
    if mode == "prompt_json":
        catalog = [x["function"] for x in TOOLS]
        base["messages"] = [
            {"role":"system", "content": (
                "Select exactly one function from the catalog. Respond ONLY with a valid JSON "
                "object containing 'name' and 'arguments'. Do not answer the task directly. "
                "Keep argument values in the form required by the request and the function. "
                "Function catalog: " + json.dumps(catalog, ensure_ascii=False)
            )},
            {"role":"user", "content":case["prompt"]},
        ]
        base["format"] = "json"
    elif mode == "native_tool":
        base["messages"] = [
            {"role":"system", "content":"Call exactly one appropriate tool. Do not answer directly."},
            {"role":"user", "content":case["prompt"]},
        ]
        base["tools"] = TOOLS
    else:
        raise ValueError("Mode tidak dikenal: " + mode)
    return base


def parse_output(response, mode):
    msg = response.get("message") or {}
    content = msg.get("content") or ""
    if mode == "prompt_json":
        try:
            parsed = json.loads(content)
        except (json.JSONDecodeError, TypeError):
            return None, "invalid_json", content
        if not isinstance(parsed, dict) or not isinstance(parsed.get("name"), str) or not isinstance(parsed.get("arguments"), dict):
            return None, "invalid_structure", content
        return {"name":parsed["name"], "arguments":parsed["arguments"]}, "parsed", content

    tool_calls = msg.get("tool_calls") or []
    if not tool_calls:
        return None, "no_native_tool_call", content
    if len(tool_calls) != 1:
        return None, "multiple_native_tool_calls", json.dumps(tool_calls, ensure_ascii=False)
    fun = (tool_calls[0] or {}).get("function") or {}
    args = fun.get("arguments", {})
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            return None, "invalid_native_arguments", json.dumps(tool_calls, ensure_ascii=False)
    if not isinstance(fun.get("name"), str) or not isinstance(args, dict):
        return None, "invalid_native_structure", json.dumps(tool_calls, ensure_ascii=False)
    return {"name":fun["name"], "arguments":args}, "parsed", json.dumps(tool_calls, ensure_ascii=False)


def score_prediction(pred, parse_status, case):
    if parse_status != "parsed":
        return parse_status
    if pred["name"] != case["expected_name"]:
        return "wrong_function"
    expected = case["expected_arguments"]
    actual = pred["arguments"]
    if set(actual) != set(expected):
        return "wrong_argument_keys"
    # Periksa kecocokan nilai argumen.
    if any(not isinstance(actual[k], str) or actual[k].strip().casefold() != expected[k].strip().casefold() for k in expected):
        return "wrong_argument_values"
    return "correct"


def test_one(model, case, mode):
    record = {
        "model":model, "mode":mode, "case_id":case["case_id"], "language":case["language"],
        "user_prompt":case["prompt"], "expected_function":case["expected_name"],
        "expected_arguments":json.dumps(case["expected_arguments"], ensure_ascii=False),
        "predicted_function":"", "predicted_arguments":"", "status":"",
        "latency_seconds":"", "error":"", "raw_model_output":"",
        "timestamp_utc":datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    start = time.perf_counter()
    try:
        res = ollama_request("/api/chat", build_payload(model, case, mode))
        record["latency_seconds"] = round(time.perf_counter() - start, 2)
        pred, parse_status, raw = parse_output(res, mode)
        record["raw_model_output"] = raw
        record["status"] = score_prediction(pred, parse_status, case)
        if pred:
            record["predicted_function"] = pred["name"]
            record["predicted_arguments"] = json.dumps(pred["arguments"], ensure_ascii=False)
    except Exception as exc:
        record["latency_seconds"] = round(time.perf_counter() - start, 2)
        record["error"] = str(exc)
        low = str(exc).lower()
        if mode == "native_tool" and ("does not support tools" in low or "does not support tool" in low):
            record["status"] = "native_unsupported_by_server"
        else:
            record["status"] = "api_error"
    return record

## 5. Pelaksanaan pengujian

Setiap model menerima empat perintah pada pengujian prompt JSON dan dua perintah pada pengujian native tool calling. Total pengujian adalah **18 permintaan**.

Model dijalankan satu per satu. Jika terjadi kesalahan pada suatu pengujian, pesan kesalahannya dicatat dan pengujian berikutnya tetap dilanjutkan.


In [6]:
results = []

for model in MODELS:
    print("\n" + "=" * 76)
    print("MODEL:", model)
    print("=" * 76)

    if RUN_JSON_MODE:
        print("[MODE A: prompt JSON]")
        for case in CASES:
            result = test_one(model, case, "prompt_json")
            results.append(result)
            print(f"  {case['case_id']:12s} {result['status']:30s} {result['latency_seconds']} s")
            if result["error"]:
                print("     Error:", result["error"][:250])

    if RUN_NATIVE_MODE:
        print("[MODE B: native tool calling]")
        for case in CASES[:2]:  # Dua perintah untuk fungsi lampu
            result = test_one(model, case, "native_tool")
            results.append(result)
            print(f"  {case['case_id']:12s} {result['status']:30s} {result['latency_seconds']} s")
            if result["error"]:
                print("     Error:", result["error"][:250])

print("\nTotal pengujian tercatat:", len(results))


MODEL: qwen2.5:7b-instruct
[MODE A: prompt JSON]
  EN_LIGHT     correct                        6.78 s
  ID_LIGHT     correct                        6.6 s
  EN_WEATHER   correct                        5.53 s
  ID_WEATHER   correct                        5.52 s
[MODE B: native tool calling]
  EN_LIGHT     correct                        5.67 s
  ID_LIGHT     correct                        5.69 s

MODEL: gemma2:9b-instruct-q4_K_M
[MODE A: prompt JSON]
  EN_LIGHT     correct                        9.96 s
  ID_LIGHT     correct                        9.67 s
  EN_WEATHER   correct                        9.39 s
  ID_WEATHER   correct                        9.35 s
[MODE B: native tool calling]
  EN_LIGHT     native_unsupported_by_server   0.23 s
     Error: HTTP 400 dari Ollama: {"error":"registry.ollama.ai/library/gemma2:9b-instruct-q4_K_M does not support tools"}
  ID_LIGHT     native_unsupported_by_server   0.0 s
     Error: HTTP 400 dari Ollama: {"error":"registry.ollama.ai/library/gemma2:

## 6. Hasil pengujian

Hasil setiap permintaan dicatat dalam tabel dan disimpan ke `results/model_selection/compatibility_smoke_test.csv`. Informasi yang disimpan meliputi nama model, bahasa, fungsi yang diprediksi, argumen, status penilaian, dan waktu respons.

Status `correct` menunjukkan kecocokan pada kasus uji awal, bukan akurasi model pada keseluruhan dataset.


In [7]:
# Tentukan folder utama proyek.
working = Path.cwd().resolve()
project_root = next(
    (p for p in (working, *working.parents) if (p / "data").is_dir() and (p / "results").is_dir()),
    working,
)
output_dir = project_root / "results" / "model_selection"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "compatibility_smoke_test.csv"

if results:
    with output_file.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    print("CSV hasil:", output_file)
else:
    print("Belum ada hasil pengujian.")

headers = ["Model", "JSON benar (dari 4)", "Native benar (dari 2)", "Status native lainnya"]
rows = []
for model in MODELS:
    json_rs = [r for r in results if r["model"] == model and r["mode"] == "prompt_json"]
    native_rs = [r for r in results if r["model"] == model and r["mode"] == "native_tool"]
    other_native = Counter(r["status"] for r in native_rs if r["status"] != "correct")
    rows.append([
        model,
        f"{sum(r['status']=='correct' for r in json_rs)}/{len(json_rs)}" if json_rs else "—",
        f"{sum(r['status']=='correct' for r in native_rs)}/{len(native_rs)}" if native_rs else "—",
        "; ".join(f"{k}: {v}" for k, v in other_native.items()) or "—",
    ])

lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
lines += ["| " + " | ".join(str(v).replace("|", "\\|") for v in row) + " |" for row in rows]
display(Markdown("\n".join(lines)))

print("\nRincian ketidaksesuaian:")
for r in results:
    if r["status"] != "correct":
        print(f"- {r['model']} | {r['mode']} | {r['case_id']} | {r['status']}")
        if r["predicted_function"]:
            print("  Prediksi:", r["predicted_function"], r["predicted_arguments"][:150])
        if r["error"]:
            print("  Error:", r["error"][:200])

CSV hasil: D:\KP-Agentic-AI\results\model_selection\compatibility_smoke_test.csv


| Model | JSON benar (dari 4) | Native benar (dari 2) | Status native lainnya |
| --- | --- | --- | --- |
| qwen2.5:7b-instruct | 4/4 | 2/2 | — |
| gemma2:9b-instruct-q4_K_M | 4/4 | 0/2 | native_unsupported_by_server: 2 |
| Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | 0/4 | 0/2 | native_unsupported_by_server: 2 |


Rincian ketidaksesuaian:
- gemma2:9b-instruct-q4_K_M | native_tool | EN_LIGHT | native_unsupported_by_server
  Error: HTTP 400 dari Ollama: {"error":"registry.ollama.ai/library/gemma2:9b-instruct-q4_K_M does not support tools"}
- gemma2:9b-instruct-q4_K_M | native_tool | ID_LIGHT | native_unsupported_by_server
  Error: HTTP 400 dari Ollama: {"error":"registry.ollama.ai/library/gemma2:9b-instruct-q4_K_M does not support tools"}
- Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | prompt_json | EN_LIGHT | invalid_structure
- Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | prompt_json | ID_LIGHT | invalid_structure
- Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | prompt_json | EN_WEATHER | invalid_structure
- Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | prompt_json | ID_WEATHER | invalid_structure
- Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | native_tool | EN_LIGHT | native_unsupported_by_server
  Error: HTTP 400 dari Ollama: {"error":"registry.ollama.ai/Sup

In [8]:

# Pemeriksaan tambahan Sahabat-AI
# Membandingkan output dengan dan tanpa mode JSON

model = MODELS[2]
case = CASES[1]  # Perintah bahasa Indonesia

print("MODEL:", model)
print("INPUT:", case["prompt"])

for use_json in [True, False]:
    payload = build_payload(model, case, "prompt_json")

    if not use_json:
        payload.pop("format", None)

    mode = "Dengan format JSON" if use_json else "Tanpa format JSON"

    print("\n" + "=" * 50)
    print("PENGUJIAN:", mode)

    try:
        response = ollama_request("/api/chat", payload)
        output = response.get("message", {}).get("content", "")

        print("OUTPUT MODEL:")
        print(output[:2000])

        try:
            parsed = json.loads(output)
            valid = (
                isinstance(parsed, dict)
                and isinstance(parsed.get("name"), str)
                and isinstance(parsed.get("arguments"), dict)
            )
            print("STRUKTUR FUNCTION CALL:", "VALID" if valid else "TIDAK VALID")
        except json.JSONDecodeError:
            print("STRUKTUR FUNCTION CALL: BUKAN JSON VALID")

    except Exception as e:
        print("ERROR:", str(e))


MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
INPUT: Tolong nyalakan lampu kamar tidur.

PENGUJIAN: Dengan format JSON
OUTPUT MODEL:
{
  "id": "1",
  "name": "John Doe",
  "email": "john.doe@example.com",
  "phone": "123-456-7890",
  "address": {
    "street": "123 Main St",
    "city": "Anytown",
    "state": "CA",
    "zip": "90210"
  }
}
STRUKTUR FUNCTION CALL: TIDAK VALID

PENGUJIAN: Tanpa format JSON
OUTPUT MODEL:


STRUKTUR FUNCTION CALL: BUKAN JSON VALID


In [9]:

# Pemeriksaan konfigurasi Sahabat-AI

model = MODELS[2]

info = ollama_request("/api/show", {"model": model})

print("=== KONFIGURASI MODEL ===")
print("Model:", model)
print("\nTemplate:")
print(info.get("template", "Tidak tersedia")[:1500])

print("\nParameters:")
print(info.get("parameters", "Tidak tersedia"))

print("\n=== UJI RESPONS SEDERHANA ===")

payload = {
    "model": model,
    "messages": [
        {
            "role": "user",
            "content": "Jawab dengan satu kata saja: Halo"
        }
    ],
    "stream": False,
    "options": {
        "temperature": 0,
        "num_ctx": 2048,
        "num_predict": 64
    },
    "keep_alive": 0
}

response = ollama_request("/api/chat", payload)

print("Respons:", repr(response.get("message", {}).get("content")))
print("Done reason:", response.get("done_reason"))
print("Token input:", response.get("prompt_eval_count"))
print("Token output:", response.get("eval_count"))


=== KONFIGURASI MODEL ===
Model: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m

Template:
{{.Input}}

Parameters:
temperature                    0.7
top_p                          0.9
stop                           "User:"
stop                           "Assistant:"

=== UJI RESPONS SEDERHANA ===
Respons: '\n'
Done reason: stop
Token input: 5
Token output: 2


In [10]:

# Uji Sahabat-AI dengan format prompt Gemma 2

model = MODELS[2]

prompt = (
    "<start_of_turn>user\n"
    "Jawab pertanyaan berikut dengan singkat. "
    "Apa ibu kota Indonesia?\n"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

payload = {
    "model": model,
    "prompt": prompt,
    "raw": True,
    "stream": False,
    "options": {
        "temperature": 0,
        "num_ctx": 2048,
        "num_predict": 128,
        "stop": ["<end_of_turn>", "<eos>"]
    },
    "keep_alive": 0
}

try:
    response = ollama_request("/api/generate", payload)

    print("MODEL:", model)
    print("RESPONS:", repr(response.get("response", "")))
    print("DONE REASON:", response.get("done_reason"))
    print("TOKEN OUTPUT:", response.get("eval_count"))

except Exception as e:
    print("ERROR:", str(e))


MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
RESPONS: 'Jakarta.'
DONE REASON: stop
TOKEN OUTPUT: 3


In [11]:

# Pengujian function calling Sahabat-AI dengan template Gemma 2

import json

model = MODELS[2]

instruksi = """
Pilih satu fungsi yang sesuai dengan permintaan pengguna.

Fungsi tersedia:
1. control_light(room: string, action: "on" atau "off")
   Untuk menyalakan atau mematikan lampu.

2. get_weather(city: string)
   Untuk memperoleh informasi cuaca.

Jawab hanya menggunakan JSON dengan struktur:
{"name": "nama_fungsi", "arguments": {"parameter": "nilai"}}

Jangan tambahkan penjelasan atau teks lainnya.

Permintaan pengguna: Tolong nyalakan lampu kamar tidur.
"""

prompt = (
    "<start_of_turn>user\n"
    + instruksi.strip()
    + "\n<end_of_turn>\n"
    + "<start_of_turn>model\n"
)

payload = {
    "model": model,
    "prompt": prompt,
    "raw": True,
    "stream": False,
    "options": {
        "temperature": 0,
        "num_ctx": 2048,
        "num_predict": 256,
        "stop": ["<end_of_turn>", "<eos>"]
    },
    "keep_alive": 0
}

response = ollama_request("/api/generate", payload)

output = response.get("response", "").strip()

print("MODEL:", model)
print("OUTPUT:", output)
print("DONE REASON:", response.get("done_reason"))

try:
    prediction = json.loads(output)

    correct = (
        prediction.get("name") == "control_light"
        and prediction.get("arguments", {}).get("room") == "kamar tidur"
        and prediction.get("arguments", {}).get("action") == "on"
    )

    print("JSON VALID: Ya")
    print("FUNCTION CALL BENAR:", correct)

except (json.JSONDecodeError, AttributeError):
    print("JSON VALID: Tidak")
    print("FUNCTION CALL BENAR: False")


MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
OUTPUT: {"name": "control_light", "arguments": {"room": "kamar tidur", "action": "on"}}
DONE REASON: stop
JSON VALID: Ya
FUNCTION CALL BENAR: True


In [12]:

# Pengujian lanjutan Sahabat-AI

test_cases = [
    ("Matikan lampu ruang tamu.", "control_light",
     {"room": "ruang tamu", "action": "off"}),

    ("Bagaimana cuaca di Bandung?", "get_weather",
     {"city": "Bandung"}),

    ("Turn on the bedroom light.", "control_light",
     {"room": "bedroom", "action": "on"})
]

model = MODELS[2]
hasil = []

for teks, fungsi, argumen in test_cases:
    instruksi = f"""
Pilih fungsi yang sesuai dengan permintaan pengguna.

Fungsi:
1. control_light(room: string, action: "on" atau "off")
2. get_weather(city: string)

Jawab hanya JSON:
{{"name": "nama_fungsi", "arguments": {{}}}}

Permintaan: {teks}
"""

    prompt = (
        "<start_of_turn>user\n"
        + instruksi.strip()
        + "\n<end_of_turn>\n"
        + "<start_of_turn>model\n"
    )

    payload = {
        "model": model,
        "prompt": prompt,
        "raw": True,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_ctx": 2048,
            "num_predict": 256,
            "stop": ["<end_of_turn>", "<eos>"]
        },
        "keep_alive": 0
    }

    try:
        response = ollama_request("/api/generate", payload)
        output = response.get("response", "").strip()
        prediksi = json.loads(output)

        benar = (
            isinstance(prediksi, dict)
            and prediksi.get("name") == fungsi
            and prediksi.get("arguments") == argumen
        )

        hasil.append(benar)

        print("\nINPUT:", teks)
        print("OUTPUT:", output)
        print("BENAR:", benar)

    except Exception as e:
        hasil.append(False)
        print("\nINPUT:", teks)
        print("ERROR:", str(e))

print("\nHASIL:", sum(hasil), "/", len(hasil))



INPUT: Matikan lampu ruang tamu.
OUTPUT: {"name": "control_light", "arguments": {"room": "ruang tamu", "action": "off"}}
BENAR: True

INPUT: Bagaimana cuaca di Bandung?
OUTPUT: {"name": "get_weather", "arguments": {"city": "Bandung"}}
BENAR: True

INPUT: Turn on the bedroom light.
OUTPUT: {"name": "control_light", "arguments": {"room": "bedroom", "action": "on"}}
BENAR: True

HASIL: 3 / 3


In [13]:

# Pengujian ulang Sahabat-AI pada empat kasus awal

model = MODELS[2]
hasil_retest = []

for case in CASES:
    konfigurasi = build_payload(model, case, "prompt_json")

    instruksi = konfigurasi["messages"][0]["content"]
    pertanyaan = konfigurasi["messages"][1]["content"]

    prompt = (
        "<start_of_turn>user\n"
        + instruksi
        + "\n\nPermintaan pengguna:\n"
        + pertanyaan
        + "\n<end_of_turn>\n"
        + "<start_of_turn>model\n"
    )

    payload = {
        "model": model,
        "prompt": prompt,
        "raw": True,
        "stream": False,
        "options": {
            **OPTIONS,
            "stop": ["<end_of_turn>", "<eos>"]
        },
        "keep_alive": 0
    }

    try:
        response = ollama_request("/api/generate", payload)
        output = response.get("response", "").strip()
        prediksi = json.loads(output)

        if (
            isinstance(prediksi, dict)
            and isinstance(prediksi.get("name"), str)
            and isinstance(prediksi.get("arguments"), dict)
        ):
            status = score_prediction(prediksi, "parsed", case)
        else:
            status = "invalid_structure"

    except json.JSONDecodeError:
        output = output if "output" in locals() else ""
        status = "invalid_json"
    except Exception as e:
        output = str(e)
        status = "api_error"

    hasil_retest.append({
        "case_id": case["case_id"],
        "status": status,
        "output": output
    })

    print("\nCASE:", case["case_id"])
    print("OUTPUT:", output)
    print("STATUS:", status)

benar = sum(r["status"] == "correct" for r in hasil_retest)

print("\n=== HASIL RETEST SAHABAT-AI ===")
print(f"Benar: {benar}/{len(hasil_retest)}")



CASE: EN_LIGHT
OUTPUT: {"name": "control_light", "arguments": {"room": "bedroom", "action": "on"}}
STATUS: correct

CASE: ID_LIGHT
OUTPUT: {"name": "control_light", "arguments": {"room": "kamar tidur", "action": "on"}}
STATUS: correct

CASE: EN_WEATHER
OUTPUT: {"name": "get_weather", "arguments": {"city": "Jakarta"}}
STATUS: correct

CASE: ID_WEATHER
OUTPUT: {"name": "get_weather", "arguments": {"city": "Jakarta"}}
STATUS: correct

=== HASIL RETEST SAHABAT-AI ===
Benar: 4/4
